In [1]:
import cv2
import mediapipe as mp
import pyautogui  
import time

In [5]:
# import pyautogui
# import time

# time.sleep(2)
# pyautogui.moveTo(500, 500, duration=2)

In [2]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1 , min_detection_confidence=0.7)


In [ ]:
# import cv2

# for index in range(5):
#     cap = cv2.VideoCapture(index, cv2.CAP_DSHOW)
#     print(f"Camera {index}: {cap.isOpened()}")
#     cap.release()

Camera 0: True
Camera 1: False
Camera 2: False
Camera 3: False
Camera 4: False


In [11]:
# cap = cv2.VideoCapture(0 , cv2.CAP_V4L2)

import math


cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

#gesture tiime control
click_start_time = None
click_times = []
click_cooldown= 0.5  # seconds
scroll_mode = False
freeze_cursor = False
screenshot_cooldown = 2.0  # seconds
last_screenshot_time = 0


#PYAUTOGUI
screen_w,screen_h = pyautogui.size()
print("\n HAND MOUSE CONTROL")
prev_screen_x , prev_screen_y = 0,0


if not cap.isOpened():
    print("Error: Could not open video stream.")
    exit()

# Set MJPEG format to handle video feed properly in WSL2
# cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))


cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)

cv2.namedWindow("Video Stream", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Video Stream", 600, 600)


while True:
    ret,frame = cap.read()
    if not ret:
        print("Error: Could not read frame.")
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)
    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            #get fiinger tips
        thumb_tips = hand_landmarks.landmark[4]
        index_tips = hand_landmarks.landmark[8]
        middle_tips = hand_landmarks.landmark[12]
        ring_tips = hand_landmarks.landmark[16]
        pinky_tips = hand_landmarks.landmark[20]

        fingers = [
            1 if hand_landmarks.landmark[tip].y < hand_landmarks.landmark[tip - 2].y else 0
            for tip in [8, 12, 16, 20]
        ]   

        #distance between thumb and index finger
    
        dist=math.hypot(thumb_tips.x - index_tips.x , thumb_tips.y - index_tips.y)
        if dist < 0.06:
            if not freeze_cursor:
                freeze_cursor = True
                click_times.append(time.time())

                #double click
                if len(click_times) >= 2 and click_times[-1] - click_times[-2] < 0.4:
                    pyautogui.doubleClick()
                    cv2.putText(frame,"Double Click",(10,50),cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,255),2)
                    click_times = []  # Reset click times after double click
                else:
                    pyautogui.click()
                    cv2.putText(frame,"Single Click",(10,50),cv2.FONT_HERSHEY_SIMPLEX,1,(255,255,0),2)
        else:
            freeze_cursor = False

        #move index finger
        if not freeze_cursor:
            screen_x = int(index_tips.x * screen_w)
            screen_y = int(index_tips.y * screen_h)
            pyautogui.moveTo(screen_x, screen_y, duration=0.05)
            prev_screen_x, prev_screen_y = screen_x, screen_y

        #scroll mode
        if sum(fingers)==4:
            scroll_mode = True
            # cv2.putText(frame,"Scroll Mode",(10,50),cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)
        else:
            scroll_mode = False

            #scroll mode actions
        if scroll_mode:
            if index_tips.y<0.3:
                pyautogui.scroll(60)
                cv2.putText(frame,"Scroll Up",(10,60),cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)
            elif index_tips.y>0.5:
                pyautogui.scroll(-60)
                cv2.putText(frame,"Scroll Down",(10,60),cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255),2)

        #screenshot mode
        if sum(fingers)==0:
            current_time = time.time()
            if current_time - last_screenshot_time > screenshot_cooldown:
                pyautogui.screenshot(f'screenshot{int(current_time)}.png')
                cv2.putText(frame,"Screenshot Taken",(10,130),cv2.FONT_HERSHEY_SIMPLEX,1,(255,0,255),2)
                last_screenshot_time = current_time




           

    cv2.imshow('Video Stream', frame)

    if cv2.waitKey(1) == ord ('q'):
        break

cap.release()
cv2.destroyAllWindows()


 HAND MOUSE CONTROL


#### Finger tips
